In [1]:
import os
from datasets import load_dataset

dataset_name = "SQuAD_2.0"

train_path = os.path.join(os.getcwd(), "Datasets", dataset_name, "train-00000-of-00001.parquet")
val_path = os.path.join(os.getcwd(), "Datasets", dataset_name, "validation-00000-of-00001.parquet")

dataset = load_dataset("parquet", data_files={'train': train_path, 'val': val_path})
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    val: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})

In [2]:
from datasets import DatasetDict

# time to reduce the dataset
# reduced_train_set = dataset["train"].shuffle(seed=42).select(range(65000))
reduced_train_set = dataset["train"].select(range(2070,2090))
reduced_val_set = dataset["val"].shuffle(seed=42).select(range(5500))
reduced_test_set = dataset["val"].shuffle(seed=42).select(range(5500, 6000))

reduced_set = DatasetDict({"train":reduced_train_set,
                           "val":reduced_val_set,
                           "test":reduced_test_set})

reduced_set

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 20
    })
    val: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 5500
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 500
    })
})

In [43]:
dataset["train"][2076]

{'id': '5a8d7bf7df8bba001a0f9ab2',
 'title': 'The_Legend_of_Zelda:_Twilight_Princess',
 'context': 'The Legend of Zelda: Twilight Princess (Japanese: ゼルダの伝説 トワイライトプリンセス, Hepburn: Zeruda no Densetsu: Towairaito Purinsesu?) is an action-adventure game developed and published by Nintendo for the GameCube and Wii home video game consoles. It is the thirteenth installment in the The Legend of Zelda series. Originally planned for release on the GameCube in November 2005, Twilight Princess was delayed by Nintendo to allow its developers to refine the game, add more content, and port it to the Wii. The Wii version was released alongside the console in North America in November 2006, and in Japan, Europe, and Australia the following month. The GameCube version was released worldwide in December 2006.[b]',
 'question': 'What consoles can be used to play Australia Twilight?',
 'answers': {'text': [], 'answer_start': []}}

In [ ]:
def count_no_answer(examples):
    counter = 0
    for answer in examples["answers"]:
        if len(answer["answer_start"]) == 0:
            counter += 1

    return counter

In [ ]:
count_no_answer(reduced_train_set)

In [ ]:
count_no_answer(reduced_val_set)

2774

In [ ]:
count_no_answer(reduced_test_set)

2762

In [3]:
from transformers import AutoTokenizer
model_name="distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [4]:
stride = 128
max_length = 384

def preprocess_train_function(examples):
    questions = [q.strip() for q in examples["question"]]
    # inputs contain tokenised questions and context in same list, separated by [SEP] token
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,              # max number of tokens for each input
        truncation="only_second",           # only truncates the second thing which is the context
        stride = stride,                    # controls the number of tokens that overlap
        return_overflowing_tokens=True,     # maps each input to a question in case of long contexts
        return_offsets_mapping=True,        # keeps track of which character index each token begins and ends at
        padding="max_length",               # pads each input to the max length
    )
    # [CLS] token at the beginning of each input, [SEP] to separate question and context

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")   # gives a list of which example each input corresponds to, each example may have multiple due to truncation
    answers = examples["answers"]
    start_positions = []
    end_positions = []
    example_ids = []

    for i, offset in enumerate(offset_mapping):
        # accounting for any overlapping
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        example_ids.append(examples["id"][sample_idx])
        # accounts for questions with no answer and use CLS token
        if len(answer["answer_start"]) == 0:
            start_char = 0
            end_char = 0
        else:
            start_char = answer["answer_start"][0]                          # get the index of the starting character
            end_char = answer["answer_start"][0] + len(answer["text"][0])   # get the index of the end character

        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        # At the question, sequence id = 0, for context sequence id = 1, else it is None
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label it (0, 0)
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        elif start_char == 0 and end_char == 0:             
            # if there is no answer and the answer is set to CLS, label (0, 0)
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:  # while before the end of context and start idx of token <= start idx of answer
                idx += 1
            start_positions.append(idx - 1)

            idx = context_start
            while idx <= context_end and offset[idx][1] <= end_char:
                idx += 1
            end_positions.append(idx - 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    inputs["example_id"] = example_ids
    return inputs

In [ ]:
from processing import preprocess_train_function, preprocess_val_function

In [5]:
# tokenized_train = reduced_set["train"].map(
#     preprocess_train_function,
#     batched=True,
#     remove_columns=reduced_set["train"].column_names,
# )

tokenized_train = dataset["train"].map(
    preprocess_train_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)
tokenized_train

Dataset({
    features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions', 'example_id'],
    num_rows: 131754
})

In [46]:
dataset["train"][2076]

{'id': '5a8d7bf7df8bba001a0f9ab2',
 'title': 'The_Legend_of_Zelda:_Twilight_Princess',
 'context': 'The Legend of Zelda: Twilight Princess (Japanese: ゼルダの伝説 トワイライトプリンセス, Hepburn: Zeruda no Densetsu: Towairaito Purinsesu?) is an action-adventure game developed and published by Nintendo for the GameCube and Wii home video game consoles. It is the thirteenth installment in the The Legend of Zelda series. Originally planned for release on the GameCube in November 2005, Twilight Princess was delayed by Nintendo to allow its developers to refine the game, add more content, and port it to the Wii. The Wii version was released alongside the console in North America in November 2006, and in Japan, Europe, and Australia the following month. The GameCube version was released worldwide in December 2006.[b]',
 'question': 'What consoles can be used to play Australia Twilight?',
 'answers': {'text': [], 'answer_start': []}}

In [ ]:
tokenized_train[6]

In [ ]:
print(tokenizer.decode(tokenized_train[0]["input_ids"][24:24]))

In [6]:
tokenized_eval = reduced_set["val"].map(
    preprocess_train_function,
    batched=True,
    remove_columns=reduced_set["val"].column_names,
)

# tokenized_eval = dataset["val"].map(
#     preprocess_train_function,
#     batched=True,
#     remove_columns=dataset["val"].column_names,
# )
tokenized_eval

Dataset({
    features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions', 'example_id'],
    num_rows: 5627
})

In [9]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers import AutoModelForQuestionAnswering
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device)

output_name = "t3_bert_full"
output_dir = os.path.join(os.getcwd(), "models", output_name)

batch_size = 16
steps_per_epoch = len(tokenized_train) // batch_size + 1
eval_steps = max(1, steps_per_epoch // 3)

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="steps",
    save_strategy="steps",  # Match this with eval
    logging_strategy="steps",
    logging_steps=eval_steps,
    eval_steps=eval_steps,
    save_steps=eval_steps,  # Add this line
    learning_rate=3e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.001,
    push_to_hub=False,
    load_best_model_at_end=True,
    # warmup_ratio = 0.2,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.01)],
)

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
trainer.train()

Step,Training Loss,Validation Loss
2745,1.773900,1.360165
5490,1.314900,1.309587
8235,1.191100,1.212596
10980,0.915400,1.218759
13725,0.899700,1.272434
16470,0.860900,1.234641
19215,0.642400,1.469858
21960,0.649900,1.431878
24705,0.628500,1.446910


TrainOutput(global_step=24705, training_loss=0.9863075106570026, metrics={'train_runtime': 4408.0644, 'train_samples_per_second': 89.668, 'train_steps_per_second': 5.605, 'total_flos': 3.873165421863629e+16, 'train_loss': 0.9863075106570026, 'epoch': 3.0})

In [3]:
max_length = 384
stride = 128

def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs

In [5]:
import os
from transformers import AutoTokenizer

output_name = "t3_bert_full"
output_dir = os.path.join(os.getcwd(), "models", output_name, "checkpoint-24705")

tokenizer = AutoTokenizer.from_pretrained(output_dir)

tokenized_test = reduced_set["test"].map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=reduced_set["test"].column_names,
)

# tokenized_test = dataset["val"].map(
#     preprocess_validation_examples,
#     batched=True,
#     remove_columns=dataset["val"].column_names,
# )
tokenized_test

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 509
})

In [63]:
dataset["val"].shuffle(seed=42)[5512]

{'id': '5ad24b60d7d075001a428be5',
 'title': 'Huguenot',
 'context': "By 1620 the Huguenots were on the defensive, and the government increasingly applied pressure. A series of three small civil wars known as the Huguenot rebellions broke out, mainly in southwestern France, between 1621 and 1629. revolted against royal authority. The uprising occurred a decade following the death of Henry IV, a Huguenot before converting to Catholicism, who had protected Protestants through the Edict of Nantes. His successor Louis XIII, under the regency of his Italian Catholic mother Marie de' Medici, became more intolerant of Protestantism. The Huguenots respond by establishing independent political and military structures, establishing diplomatic contacts with foreign powers, and openly revolting against central power. The rebellions were implacably suppressed by the French Crown.[citation needed]",
 'question': 'In what year was Louis XIII crowned?',
 'answers': {'text': [], 'answer_start': []}}

In [ ]:
tokenized_test[19]["offset_mapping"]

In [6]:
import torch
from transformers import AutoModelForQuestionAnswering

t_tokenized_test = tokenized_test.remove_columns(["example_id", "offset_mapping"])
t_tokenized_test.set_format("torch")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained_model = AutoModelForQuestionAnswering.from_pretrained(output_dir).to(device)

batch = {k: t_tokenized_test[k].to(device) for k in t_tokenized_test.column_names}

with torch.no_grad():
    outputs = trained_model(**batch)

In [ ]:
from torch.utils.data import DataLoader
import torch
from transformers import AutoModelForQuestionAnswering
import torch.nn.functional as F

t_tokenized_test = tokenized_test.remove_columns(["offset_mapping"])
t_tokenized_test.set_format("torch")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained_model = AutoModelForQuestionAnswering.from_pretrained(output_dir).to(device)

batch_size = 64  # Adjust this based on your GPU memory
test_dataloader = DataLoader(t_tokenized_test, batch_size=batch_size)

start_probs = []
end_probs = []

# trained_model.eval()  # Set the model to evaluation mode
# with torch.no_grad():
#     for batch in test_dataloader:
#         batch = {k: v.to(device) for k, v in batch.items()}
#         outputs = trained_model(**batch)
#         start_logits = F.softmax(outputs.start_logits, dim=-1).cpu().numpy()
#         end_logits = F.softmax(outputs.end_logits, dim=-1).cpu().numpy()

#         start_probs.extend(start_logits)
#         end_probs.extend(end_logits)

# Now all_start_logits and all_end_logits contain the predictions for your entire test set

In [12]:
from torch.utils.data import DataLoader
import torch
from transformers import AutoModelForQuestionAnswering
import torch.nn.functional as F

t_tokenized_test = tokenized_test.remove_columns(["example_id", "offset_mapping"])
t_tokenized_test.set_format("torch")

n_best = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained_model = AutoModelForQuestionAnswering.from_pretrained(output_dir).to(device)

batch_size = 32  # Adjust this based on your GPU memory
test_dataloader = DataLoader(t_tokenized_test, batch_size=batch_size, shuffle=False)    # no need for shuffling

start_probs = torch.empty((len(t_tokenized_test), 384), device=device)
end_probs = torch.empty((len(t_tokenized_test), 384), device=device)

# best_start_probs = torch.empty((len(t_tokenized_test), n_best), device=device)
# best_end_probs = torch.empty((len(t_tokenized_test), n_best), device=device)

trained_model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    for i, batch in enumerate(test_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = trained_model(**batch)

        batch_start_probs = F.softmax(outputs.start_logits, dim=-1)
        batch_end_probs = F.softmax(outputs.end_logits, dim=-1)

        start_idx = i * batch_size
        end_idx = start_idx + batch_start_probs.shape[0]

        start_probs[start_idx:end_idx] = batch_start_probs
        end_probs[start_idx:end_idx] = batch_end_probs


In [13]:
no_answer_probs = start_probs[:, 0] * end_probs[:, 0]

start_indexes = torch.argsort(start_probs, dim=-1, descending=True)[:, :n_best] # get n best scores for each example
end_indexes = torch.argsort(end_probs, dim=-1, descending=True)[:, :n_best]

best_start_probs = torch.gather(start_probs, dim=1, index=start_indexes)
best_end_probs = torch.gather(end_probs, dim=1, index=end_indexes)

best_probs = best_start_probs.unsqueeze(2) * best_end_probs.unsqueeze(1)    # calcalate the n_best squared best probabilities for each feature

max_prob_indices = torch.argsort(best_probs.view((len(t_tokenized_test), -1)), dim=1, descending=True)
# indexes of a flattened best probs
# start idx can be found idx // n_best
# end idx can be found idx % n_best
# get the token index from start_indexes


In [7]:
import collections
example_to_features = collections.defaultdict(list)
for idx, feature in enumerate(tokenized_test):
    example_to_features[feature["example_id"]].append(idx)


In [15]:
from tqdm import tqdm

max_answer_length = 30
predicted_answers = []
no_answer_threshold = 0.05

for example in tqdm(reduced_set["test"]):
    example_id = example["id"]
    context = example["context"]
    answers = []    # track all answers for this id

    # go through each feature that is associated to that example
    for feature_index in example_to_features[example_id]:
        offsets = tokenized_test["offset_mapping"][feature_index]   # get offset mapping
        probability_indicies = max_prob_indices[feature_index]      # get n_best squared indexes

        no_answer_probability = no_answer_probs[feature_index].cpu().item()

        # if the no answer probability exceeds threshold, give an empty answer
        if no_answer_probability > no_answer_threshold:
            answers.append(
                {
                    "text": "",
                    "prob_score": no_answer_probability,
                    "no_answer_probability": no_answer_probability,
                }
            )
            break
        
        # iterate through the indicies, going from maximum probability first
        for flat_idx in probability_indicies:
            start_index = start_indexes[feature_index, flat_idx // n_best]
            end_index = end_indexes[feature_index, flat_idx % n_best]

            # if the highest probability index is not part of context then move to next highest probability
            if offsets[start_index] is None or offsets[end_index] is None:
                continue
            if (end_index < start_index or end_index - start_index + 1> max_answer_length):
                continue
            # only append one set of probabilities per feature, where the prob score is the highest
            answers.append(
                {
                    "text": context[offsets[start_index][0] : offsets[end_index][1]],
                    "prob_score": best_probs[feature_index, flat_idx // n_best, flat_idx % n_best].cpu().item(),
                    "no_answer_probability": no_answer_probability,
                }
            )
            break

    if len(answers) == 0:
        predicted_answers.append({"id": example_id, "prediction_text": "", "no_answer_probability": 1.0})
    else:
        # check for the best answer for each id
        best_answer = max(answers, key=lambda x: x["prob_score"])
        predicted_answers.append({"id": example_id, "prediction_text": best_answer["text"], "no_answer_probability": best_answer["no_answer_probability"]})
        

100%|██████████| 500/500 [04:09<00:00,  2.01it/s]


In [ ]:
import torch.nn.functional as F

start_probs = F.softmax(outputs.start_logits, dim=-1).cpu().numpy()
end_probs = F.softmax(outputs.end_logits, dim=-1).cpu().numpy()

In [10]:
start_probs = outputs.start_logits.cpu().numpy()
end_probs = outputs.end_logits.cpu().numpy()

In [11]:
# small_eval_set = reduced_set["test"]
# eval_Set = tokenised_test
import numpy as np
from tqdm import tqdm
import collections

n_best = 5
max_answer_length = 30
predicted_answers = []
no_answer_threshold = 0.8

example_to_features = collections.defaultdict(list)
for idx, feature in enumerate(tokenized_test):
    example_to_features[feature["example_id"]].append(idx)

for example in tqdm(reduced_set["test"]):
    example_id = example["id"]
    context = example["context"]
    answers = []    # track all answers for this id

    # for each input that corresponds to that example
    for feature_index in example_to_features[example_id]:
        start_prob = start_probs[feature_index]
        end_prob = end_probs[feature_index]
        offsets = tokenized_test["offset_mapping"][feature_index]

        # get the n highest logit values
        start_indexes = np.argsort(start_prob)[-1 : -n_best - 1 : -1].tolist()
        end_indexes = np.argsort(end_prob)[-1 : -n_best - 1 : -1].tolist()

        # no answer is probability that CLS is start and end token
        no_answer_probability = start_prob[0] + end_prob[0]

        for start_index in start_indexes:
            for end_index in end_indexes:
                # Skip answers that are not fully in the context
                if offsets[start_index] is None or offsets[end_index] is None:
                    continue
                # Skip answers with a length that is either < 0 or > max_answer_length.
                if (end_index < start_index or end_index - start_index + 1> max_answer_length
                ):
                    continue

                answers.append(
                    {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "prob_score": start_prob[start_index] + end_prob[end_index],
                        "no_answer_probability": no_answer_probability,
                    }
                )
    
    if len(answers)!= 0:
        # check for the best answer for each id
        best_answer = max(answers, key=lambda x: x["prob_score"])
        predicted_answers.append({"id": example_id, "prediction_text": best_answer["text"], "no_answer_probability": best_answer["no_answer_probability"]})
    else:
        predicted_answers.append({"id": example_id, "prediction_text": "", "no_answer_probability": 1.0})

100%|██████████| 500/500 [04:22<00:00,  1.90it/s]


In [ ]:
import numpy as np

n_best = 20
max_answer_length = 30
predicted_answers_2 = []

for example in reduced_set["test"]:
    example_id = example["id"]
    context = example["context"]
    answers = []    # track all answers for this id

    for feature_index in example_to_features[example_id]:
        start_prob = start_probs[feature_index]
        end_prob = end_probs[feature_index]
        offsets = tokenized_test["offset_mapping"][feature_index]

        start_indexes = np.argsort(start_prob)[-1 : -n_best - 1 : -1]
        end_indexes = np.argsort(end_prob)[-1 : -n_best - 1 : -1]

        no_answer_probability = start_prob[0] * end_prob[0]

        # filter out values that equal none
        valid_start_indexes = [idx for idx in start_indexes if offsets[idx] is not None]
        valid_end_indexes = [idx for idx in end_indexes if offsets[idx] is not None]

        if not valid_start_indexes or not valid_end_indexes:
            continue  # Skip if no valid start or end indices

        start_indices_grid, end_indices_grid = np.meshgrid(valid_start_indexes, valid_end_indexes)
        flat_start_indices = start_indices_grid.flatten()
        flat_end_indices = end_indices_grid.flatten()

        # Filter by length
        length_mask = np.logical_and(
            flat_end_indices >= flat_start_indices,
            flat_end_indices - flat_start_indices + 1 <= max_answer_length,
        )

        final_start_indices = flat_start_indices[length_mask]
        final_end_indices = flat_end_indices[length_mask]

        # Calculate probability scores
        final_prob_scores = start_prob[final_start_indices] * end_prob[final_end_indices]

        max_idx = np.argmax(final_prob_scores)

        answers.append(
            {
                "text": context[offsets[final_start_indices[max_idx]][0] : offsets[final_end_indices[max_idx]][1]],
                "score": final_prob_scores[max_idx],
                "no_answer_probability": no_answer_probability,
            }
        )

    # check for the best answer for each id
    best_answer = max(answers, key=lambda x: x["score"])
    predicted_answers_2.append({"id": example_id, "prediction_text": best_answer["text"], "no_answer_probability": best_answer["no_answer_probability"]})

In [12]:
references = [{"id": ex["id"], "answers": ex["answers"]} for ex in reduced_set["test"]]

In [ ]:
import evaluate
metric = evaluate.load("squad_v2")
metric.compute(predictions=predicted_answers, references=references)

{'exact': 53.0,
 'f1': 54.82032967032967,
 'total': 500,
 'HasAns_exact': 16.049382716049383,
 'HasAns_f1': 19.794917017139237,
 'HasAns_total': 243,
 'NoAns_exact': 87.93774319066148,
 'NoAns_f1': 87.93774319066148,
 'NoAns_total': 257,
 'best_exact': 59.4,
 'best_exact_thresh': 7.393261909484863,
 'best_f1': 62.94003153368374,
 'best_f1_thresh': 8.326717376708984}

In [ ]:
import evaluate
from torch.nn.functional import softmax
import torch

metric = evaluate.load("squad_v2")
# needs to be fixed group data into collections and other bs
def compute_metrics(p):
    predictions = p.predictions
    label_ids = p.label_ids

    # Convert to list of answers
    formatted_predictions = []
    formatted_references = []
    
    for idx, (pred, ref) in enumerate(zip(predictions, label_ids)):
        # Get start and end logits and no_answer probability
        start_logits = pred[0]
        end_logits = pred[1]
        no_answer_prob = pred[2]  # no_answer probability from model
        
        # Extract the span of the answer
        start_idx = torch.argmax(torch.tensor(start_logits)).item()
        end_idx = torch.argmax(torch.tensor(end_logits)).item()
        prediction_text = tokenizer.decode(
            tokenized_eval["input_ids"][start_idx:end_idx+1],
            skip_special_tokens=True
        )

        formatted_predictions.append({
            "id": tokenized_eval["example_id"],
            "prediction_text": prediction_text,
            "no_answer_probability": no_answer_prob
        })
        
        formatted_references.append({
            "id": tokenized_eval["example_id"],
            "answers": dataset["validation"][idx]["answers"]
        })
    
    result = metric.compute(predictions=formatted_predictions, references=formatted_references)
    
    return {
        "exact_match": result["exact"],
        "f1": result["f1"]
    }
